# Model Evaluation: Classification

## Overview

Classification evaluation requires more care than regression because different errors have different costs, class imbalance distorts simple metrics, and threshold choice affects all count-based metrics.

**Metric decision tree:**
```
Imbalanced classes?
  Yes -> PR-AUC, F1 per class, confusion matrix
  No  -> Accuracy, ROC-AUC acceptable
Probabilities needed?
  Yes -> Calibration plot + Brier score
  No  -> ROC-AUC, F1
Asymmetric error costs?
  Yes -> Choose threshold from PR curve
  No  -> Default threshold (0.5) may be acceptable
```

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (roc_auc_score, average_precision_score,
    brier_score_loss, f1_score, RocCurveDisplay,
    precision_recall_curve, ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve
from scipy.special import expit

rng = np.random.default_rng(42)
n = 500
X = rng.normal(0, 1, (n, 5))
true_coef = np.array([1.5, -1.2, 0.8, 0.0, 0.0])
log_odds = X @ true_coef - 0.5
label = (expit(log_odds) > 0.5).astype(int)
print(f"Prevalence: {label.mean():.3f}")
X_tr, X_te, y_tr, y_te = train_test_split(X, label, test_size=0.25,
                                            stratify=label, random_state=42)

---
## Comparing Multiple Models

In [ ]:
models = {
    "Logistic Regression": Pipeline([("sc",StandardScaler()),("lr",LogisticRegression())]),
    "Random Forest":       RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "Gradient Boosting":   HistGradientBoostingClassifier(max_iter=300, random_state=42),
    "Naive Bayes":         GaussianNB(),
}
results = []
for name, model in models.items():
    model.fit(X_tr, y_tr)
    prob = model.predict_proba(X_te)[:,1]
    results.append({
        "Model":   name,
        "AUC-ROC": roc_auc_score(y_te, prob),
        "PR-AUC":  average_precision_score(y_te, prob),
        "Brier":   brier_score_loss(y_te, prob),
        "F1":      f1_score(y_te, model.predict(X_te)),
    })
print(pd.DataFrame(results).set_index("Model").round(3))

---
## ROC and Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,5))
colors = ["#e74c3c","steelblue","#4fffb0","orange"]
for (name, model), color in zip(models.items(), colors):
    prob = model.predict_proba(X_te)[:,1]
    # ROC
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_te, prob)
    auc = roc_auc_score(y_te, prob)
    axes[0].plot(fpr, tpr, color=color, lw=1.8, label=f"{name} ({auc:.3f})")
    # PR
    prec, rec, _ = precision_recall_curve(y_te, prob)
    ap = average_precision_score(y_te, prob)
    axes[1].plot(rec, prec, color=color, lw=1.8, label=f"{name} ({ap:.3f})")
axes[0].plot([0,1],[0,1],"k--"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curves (AUC)"); axes[0].legend(fontsize=8)
axes[1].axhline(label.mean(), color="grey", linestyle="--", label="No-skill")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curves (AP)"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

---
## Calibration Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
for (name, model), color in zip(models.items(), colors):
    prob = model.predict_proba(X_te)[:,1]
    frac, mean_pred = calibration_curve(y_te, prob, n_bins=8)
    ax.plot(mean_pred, frac, "o-", color=color, lw=1.5, label=name)
ax.plot([0,1],[0,1],"k--", lw=1, label="Perfect")
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Fraction positive")
ax.set_title("Calibration Plots"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
for name, model in models.items():
    b = brier_score_loss(y_te, model.predict_proba(X_te)[:,1])
    print(f"Brier score {name}: {b:.4f}  (lower is better; 0=perfect, 0.25=no-skill)")

---
## Cross-Validated Model Comparison

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in models.items():
    auc = cross_val_score(model, X, label, cv=cv, scoring="roc_auc")
    f1  = cross_val_score(model, X, label, cv=cv, scoring="f1")
    cv_results[name] = {"AUC mean":auc.mean(),"AUC std":auc.std(),
                         "F1 mean":f1.mean(), "F1 std":f1.std()}
print(pd.DataFrame(cv_results).T.round(3))

---

## Common Pitfalls

**1. Comparing models on test RMSE/accuracy after iterating on test performance**  
Once a test set metric has guided model selection, it is no longer an unbiased estimate of generalisation. Report cross-validated performance for model selection and reserve the test set for a single final evaluation.

**2. Using ROC-AUC alone for imbalanced outcomes**  
ROC-AUC evaluates across all thresholds, weighting each equally. For rare outcomes, most useful thresholds cluster near 0. Precision-Recall AUC (average precision) focuses on the region that matters and is more informative for imbalanced classes.

**3. Not reporting Brier score when probabilities matter**  
AUC measures discrimination (ranking), not calibration. A model with AUC=0.90 may assign probabilities of 0.9 where the true rate is 0.4. Always report the Brier score and calibration plot when predicted probabilities will be used in decisions.

**4. Treating cross-validated F1 as directly comparable across datasets**  
F1 depends on the positive prevalence in each fold. If class balance differs across folds (e.g. after stratification is imperfect), fold-to-fold variance in F1 reflects sampling variation as much as model performance.

**5. Selecting a model based on a single metric**  
No single metric captures all evaluation dimensions. A model with the best AUC may have poor calibration; the model with best F1 may be overconfident. Report a metric table (AUC, PR-AUC, Brier, F1) and select based on the most operationally relevant criterion for the specific application.
---
*python_methods_library - Samantha McGarrigle*